In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")

In [0]:
bronze_df = spark.read.table(f"dbw_proj02_hospital.bronze.{table_name}").filter(f"landing_timestamp='{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as F

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

In [0]:
target_table = f"dbw_proj02_hospital.bronze.{table_name}"

if "update_timestamp" not in spark.table(target_table).columns:
    spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS (update_timestamp TIMESTAMP)")

updated_df.write \
    .mode("overwrite") \
    .saveAsTable(target_table)